# Cascade v5 — Colab / Jupyter

**Colab:** Runtime → **GPU** → jalankan **semua sel dari atas** (jangan loncat).

**MODAL_PER_TRADE = 5.0 USD** (jangan diubah)


## 0a. Colab — clone / update repo (WAJIB sel pertama)


In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/cascade_v5") if IN_COLAB else Path.cwd()
if not (ROOT / "config.py").exists() and (ROOT / "notebooks").exists():
    ROOT = ROOT.parent if (ROOT.parent / "config.py").exists() else ROOT

if IN_COLAB:
    import subprocess
    if not (ROOT / "config.py").exists():
        subprocess.call([
            "git", "clone", "--depth", "1",
            "https://github.com/heathclif-cyber/cascade_v5.git",
            str(ROOT),
        ])
    else:
        subprocess.call(["git", "-C", str(ROOT), "pull", "origin", "master"])
    # Cek patch bug CV (wajib ada di repo terbaru)
    shared = (ROOT / "pipeline" / "shared.py").read_text(encoding="utf-8")
    if "_build_purged_folds_ordinal" not in shared:
        raise RuntimeError(
            "Repo masih versi lama. Runtime -> Restart -> jalankan sel ini lagi, "
            "atau hapus folder /content/cascade_v5 lalu clone ulang."
        )
    print("Colab repo OK:", ROOT)
else:
    print("Mode lokal, ROOT =", ROOT.resolve())


## 0b. Konfigurasi — edit di sini


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ROOT & IN_COLAB sudah dari sel 0a
if not IN_COLAB:
    _here = Path.cwd()
    ROOT = _here if (_here / "config.py").exists() else _here.parent

# --- koin ---
# Pilot cepat (Colab): 3 koin | Full: None → pakai TRAINING_COINS di 04
PILOT_COINS = ["SOLUSDT", "ETHUSDT", "BNBUSDT"]
USE_PILOT = True  # False = --all di fetch/clean/engineer

# --- langkah pipeline (True/False) ---
RUN_SETUP = True
RUN_FETCH_CLEAN_ENGINEER = True
RUN_TRAIN_LGBM = True
RUN_LSTM_PIPELINE = True  # 05a → 05b → 05d → 05c
RUN_GUARDIAN = True
RUN_HOLDOUT = False       # butuh data holdout 01-03 dulu
RUN_REPORTS = True

# --- holdout ---
RUN_HOLDOUT_FETCH = False  # 01-03 --holdout
RUN_ID = "jupyter_run"

COINS_ARG = " ".join(PILOT_COINS) if USE_PILOT else ""
ALL_FLAG = "" if USE_PILOT else "--all"
HOLDOUT_FLAG = "--holdout" if RUN_HOLDOUT_FETCH else ""

print("IN_COLAB:", IN_COLAB)
print("ROOT:", ROOT.resolve())


## 1. Bootstrap environment


In [ ]:
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if RUN_SETUP:
    if IN_COLAB:
        from tools.colab_bootstrap import setup_colab

        setup_colab(repo_root=ROOT, mount_drive=False, install_deps=True)
    else:
        req = ROOT / "requirements-colab.txt"
        if req.exists():
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)]
            )
        import torch
        from core.utils import get_lstm_device

        print("Mode: lokal / Jupyter")
        print("LSTM device:", get_lstm_device())
        print("CUDA:", torch.cuda.is_available())

from config import COLAB_QUICK_COINS, TRAINING_COINS, MODAL_PER_TRADE

print("MODAL_PER_TRADE:", MODAL_PER_TRADE)
print("Coins:", PILOT_COINS if USE_PILOT else f"ALL ({len(TRAINING_COINS)})")


## 2. Helper — jalankan perintah pipeline


In [ ]:
def run_inprocess(rel_path: str, *argv: str) -> None:
    """Jalankan skrip pipeline di kernel yang sama (log tampil di Colab)."""
    import runpy

    path = ROOT / rel_path
    if not path.exists():
        raise FileNotFoundError(f"Tidak ada: {path}")

    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    if IN_COLAB:
        os.environ["CASCADE_COLAB"] = "1"
    os.environ["PYTHONUNBUFFERED"] = "1"

    old_argv = sys.argv[:]
    sys.argv = [str(path)] + list(argv)
    print("\n" + "=" * 60)
    print("IN-PROCESS", rel_path, " ".join(argv))
    print("ROOT =", ROOT.resolve())
    print("=" * 60)
    try:
        runpy.run_path(str(path), run_name="__main__")
    except SystemExit as e:
        code = e.code if e.code is not None else 0
        if code != 0:
            raise RuntimeError(f"{rel_path} gagal (exit {code})") from e
    finally:
        sys.argv = old_argv


def pipeline_argv() -> list[str]:
    """Argumen --coins / --all (+ opsional --holdout)."""
    args = (["--coins"] + PILOT_COINS) if USE_PILOT else ["--all"]
    if HOLDOUT_FLAG.strip():
        args.append("--holdout")
    return args


def run_audit() -> None:
    coins = PILOT_COINS if USE_PILOT else TRAINING_COINS
    run_inprocess("tools/audit_pipeline.py", "--coins", *coins)


## 3. Data — fetch, clean, engineer


In [ ]:
if RUN_FETCH_CLEAN_ENGINEER:
    pargs = pipeline_argv()
    run_inprocess("pipeline/01_fetch.py", *pargs)
    run_inprocess("pipeline/02_clean.py", *pargs)
    run_inprocess("pipeline/03_engineer.py", *pargs)
    from config import LABEL_DIR

    print("LABEL_DIR =", LABEL_DIR.resolve())
    labeled = list(LABEL_DIR.glob("*_h4_lgbm.parquet"))
    print("labeled files:", [p.name for p in labeled] or "(KOSONG)")
    run_audit()
    if not labeled:
        raise RuntimeError(
            "Engineer selesai tapi tidak ada *_h4_lgbm.parquet. "
            "Cek log fetch/clean di atas (harus ada SELESAI 3/3 dan Done 3/3)."
        )


## 4. Train LGBM (5-class, purged CV)


In [ ]:
def _preflight_lgbm_data() -> None:
    from config import LABEL_DIR
    files = list(LABEL_DIR.glob("*_h4_lgbm.parquet"))
    if not files:
        raise RuntimeError(
            "Tidak ada data/labeled/*_h4_lgbm.parquet — "
            "jalankan sel 3 (fetch/clean/engineer) dulu."
        )
    print(f"Data OK: {len(files)} file parquet")


if RUN_TRAIN_LGBM:
    _preflight_lgbm_data()
    run_inprocess("pipeline/04_train_lgbm.py", "--all")


## 5. LSTM — labels, sequences, OOF residual, train


In [ ]:
if RUN_LSTM_PIPELINE:
    ca = pipeline_argv()
    run_inprocess("pipeline/05a_momentum_labels.py", *ca)
    run_inprocess("pipeline/05b_build_sequences.py", *ca)
    run_inprocess("pipeline/05d_oof_residuals.py", "--all")
    run_inprocess("pipeline/05c_train_momentum_expert.py", "--all", "--run-id", RUN_ID)


## 6. Guardian v3.5


In [ ]:
if RUN_GUARDIAN:
    run_inprocess("pipeline/06_train_guardian.py", *pipeline_argv())


## 7. Holdout backtest (opsional)


In [ ]:
if RUN_HOLDOUT:
    if RUN_HOLDOUT_FETCH:
        run_inprocess("pipeline/01_fetch.py", "--all", "--holdout")
        run_inprocess("pipeline/02_clean.py", "--all", "--holdout")
        run_inprocess("pipeline/03_engineer.py", "--all", "--holdout")
    args = pipeline_argv() + ["--run-id", RUN_ID]
    run_inprocess("pipeline/07_holdout_backtest.py", *args)


## 8. Laporan — benchmark + overfitting


In [ ]:
if RUN_REPORTS:
    for tool, targv in [
        ("tools/benchmark_plan.py", []),
        ("tools/overfitting_report.py", []),
    ]:
        try:
            run_inprocess(tool, *targv)
        except RuntimeError:
            print(f"(skip {tool})")
    if (ROOT / "models" / "runs").exists():
        runs = sorted((ROOT / "models" / "runs").glob("holdout_*"))
        if runs:
            try:
                run_inprocess(
                    "tools/overfitting_report.py",
                    "--holdout-run",
                    str(runs[-1].relative_to(ROOT)),
                )
            except RuntimeError:
                pass


## 9. Simpan ke Google Drive (Colab, opsional)


In [ ]:
if IN_COLAB and False:  # ubah ke True untuk backup
    from google.colab import drive

    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/cascade_v5_backup"
    subprocess.call(f"mkdir -p {dest} && cp -r data models reports {dest}", shell=True)
    print("Backup ke", dest)


## 10. Cek artefak


In [ ]:
from pathlib import Path

artifacts = [
    "models/lgbm_tabular.pkl",
    "models/lgbm_cv_results.json",
    "models/lstm_momentum_expert.pt",
    "models/lstm_cv_results.json",
    "models/guardian_best.pkl",
]
for rel in artifacts:
    p = ROOT / rel
    tag = "OK" if p.exists() else "MISSING"
    print(f"[{tag}] {rel}")

reports = sorted((ROOT / "reports").glob("overfitting_*.md")) if (ROOT / "reports").exists() else []
if reports:
    print("\nLaporan overfitting terbaru:", reports[-1])
